# Human and LLM judgment comparison

The historical (550 rows) and unified (671 rows) forms are analyzed separately. Scores and approval percentages use thresholds of 50 and 60.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/judged_log.csv').is_file() and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from analysis.human_llm_statistics import run
from IPython.display import Image, Markdown, display


## LLM and cohort overview


In [ ]:
tables = run(ROOT)
for title, name in [
    ('LLM decisions', 'llm_overview'),
    ('LLM votes', 'llm_votes'),
    ('LLM threshold policies', 'llm_policy_50_60'),
    ('LLM provider statistics', 'llm_provider_statistics'),
    ('Historical cohort (550 judgments)', 'historical/summary'),
    ('Unified cohort (671 judgments)', 'unified/summary'),
]:
    display(Markdown(f'### {title}'), tables[name])


## Threshold comparisons

Score, approval percentage, and score with at least 50% human approval are separate decision rules. TP, TN, FP and FN use the human rule as the reference. Agreement and Cohen kappa describe concordance.


In [ ]:
for cohort in ('historical', 'unified'):
    display(Markdown(f'### {cohort.title()} threshold decisions'), tables[f'{cohort}/comparison_50_60'])


## Approval rates

These charts follow the threshold tables, as in the original analysis.


In [ ]:
for cohort in ('historical', 'unified'):
    path = ROOT / 'results/human_llm' / cohort / 'approval_50_60.png'
    display(Markdown(f'### {cohort.title()} approval rates'), Image(filename=str(path), width=900))


## Confusion matrices

Rows show the human decision; columns show the official LLM decision. The joint score and approval rule reproduces the 596 confirmed and 75 unconfirmed unified judgments at threshold 50.


In [ ]:
for cohort in ('historical', 'unified'):
    display(Markdown(f'### {cohort.title()} confusion matrices'))
    for measure in ('human_score', 'human_approval_pct', 'human_score_and_approval_pct_ge_50'):
        for threshold in (50, 60):
            path = ROOT / 'results/human_llm' / cohort / 'figures' / f'confusion_{measure}_{threshold}.png'
            display(Image(filename=str(path), width=500))


## Form level results and join checks

Unified form rows are reconstructed from `individual_responses_json`; each form and event is aggregated once.


In [ ]:
for cohort in ('historical', 'unified'):
    display(Markdown(f'### {cohort.title()} join audit'), tables[f'{cohort}/match_audit'])
    display(Markdown(f'### {cohort.title()} by form'), tables[f'{cohort}/by_form_50_60'])
    display(Markdown(f'### {cohort.title()} by category'), tables[f'{cohort}/by_category_50_60'])


## Human and LLM scores

Score comparisons follow the form and category results.


In [ ]:
for cohort in ('historical', 'unified'):
    path = ROOT / 'results/human_llm' / cohort / 'score_comparison.png'
    display(Markdown(f'### {cohort.title()} score comparison'), Image(filename=str(path), width=600))


## Output

Tables and figures are saved under `results/human_llm/historical/` and `results/human_llm/unified/`. The 35 unseen generation events form a separate experiment.
